In [1]:
import json
import random
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

MODEL = "Qwen/Qwen2-1.5B-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

message = [
    {"role": "user",
     "content": "Explain transformers in one paragraph."}
]

inputs = tokenizer.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
tokens = [100, 200, 300, 400, 500]
input_len = inputs.shape[1]
import time

def get_output_and_time(token):
    
    torch.cuda.synchronize()
    
    start = time.perf_counter()
    outputs = model.generate(inputs, min_new_tokens=token, max_new_tokens=token)
    torch.cuda.synchronize()
    
    output = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
    )
    
    elapsed = time.perf_counter() - start
    
    new_tokens = outputs.shape[1] - input_len
    tokens_per_sec = new_tokens / elapsed
    
    print("="*10)
    print(f"For tokens {token}:")
    
    print(f"Output: {output}")
    print(f"Generated tokens: {new_tokens}")
    print(f"Time: {elapsed:.3f}s")
    print(f"Tokens/sec: {tokens_per_sec:.2f}")    


In [9]:
for token in tokens:
    get_output_and_time(token)

For tokens 100:
Output: system
You are a helpful assistant.
user
Explain transformers in one paragraph.
assistant
Transformers are a type of neural network architecture used for natural language processing tasks, particularly for tasks such as machine translation and text generation. They are designed to model long-range dependencies between words or other units of input data, which is important for understanding the meaning behind complex sentences or texts. The core component of a transformer is an encoder-decoder architecture that consists of two parts: an encoder that processes the input sequence and generates a representation of it, and a decoder that takes this representation and produces a sequence
Generated tokens: 100
Time: 53.151s
Tokens/sec: 1.88
For tokens 200:
Output: system
You are a helpful assistant.
user
Explain transformers in one paragraph.
assistant
Transformers are neural network architectures used for natural language processing tasks such as machine translation, 

Qwen2-1.5B-Instruct gets on an average 1.9 tokens per second.
Now let's see about logits.

First let's analyze the primary dataset to be used.

In [2]:
from datasets import load_dataset

arc_dataset = load_dataset("allenai/ai2_arc", "ARC-Challenge")
arc_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 299
    })
})

In [3]:
train_set = arc_dataset['train']
test_set = arc_dataset['test']
validation_set = arc_dataset['validation']

import pandas as pd

def dataset_to_dataframe(set):
    
    rows = []
    
    skipped = 0
    
    for row in set:
        
        choices = row["choices"]["text"]
        
        if len(choices) < 4:
            skipped += 1
            continue
        
        rows.append({
            "question": row["question"],
            "option_A": choices[0],
            "option_B": choices[1],
            "option_C": choices[2],
            "option_D": choices[3],
            "answer_key": row["answerKey"]
        })
        
    return pd.DataFrame(rows), skipped

train_set, skipped_train = dataset_to_dataframe(train_set)
test_set, skipped_test = dataset_to_dataframe(test_set)
validation_set, skipped_val = dataset_to_dataframe(validation_set)

print(f"Total skipped: {skipped_train + skipped_test + skipped_val}")

Total skipped: 8


In [4]:
full_set = pd.concat([train_set, test_set, validation_set], ignore_index=True)

full_set.head(5)

,question,option_A,option_B,option_C,option_D,answer_key
0,George wants to warm his hands quickly by rubb...,dry palms,wet palms,palms covered with oil,palms covered with lotion,A
1,Which of the following statements best explain...,The refrigerator door is smooth.,The refrigerator door contains iron.,The refrigerator door is a good conductor.,The refrigerator door has electric wires in it.,B
2,A fold observed in layers of sedimentary rock ...,cooling of flowing magma.,converging of crustal plates.,deposition of river sediments.,solution of carbonate minerals.,B
3,Which of these do scientists offer as the most...,worldwide disease,global mountain building,rise of mammals that preyed upon plants and an...,impact of an asteroid created dust that blocke...,D
4,A boat is acted on by a river current flowing ...,west,east,north,south,B


In [5]:
total_text_len = 0

for i, row in full_set.iterrows():
    total_text_len += len(row["question"])
    total_text_len += len(row["option_A"])
    total_text_len += len(row["option_B"])
    total_text_len += len(row["option_C"])
    total_text_len += len(row["option_D"])
    
print(f"Total Text len: {total_text_len}")
print(f"Average Question len: {(total_text_len / full_set.shape[0]):.2f}")

Total Text len: 632781
Average Question len: 245.07


On average, model will be passed 245.07 characters + X(prompt)

In [6]:
PROMPT = (
    "Select only one option from A, B, C, or D to answer the following "
    "multiple-choice question. Do not output anything else.\n"
    "Question: {question}\n"
    "A. {a}\n"
    "B. {b}\n"
    "C. {c}\n"
    "D. {d}\n"
    "Answer:"
)

def build_prompt(row):
    
    return PROMPT.format(
        question=row["question"],
        a=row["option_A"], b=row["option_B"],
        c=row["option_C"], d=row["option_D"],
    )

print(build_prompt(full_set.iloc[0]))    

Select only one option from A, B, C, or D to answer the following multiple-choice question. Do not output anything else.
Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
A. dry palms
B. wet palms
C. palms covered with oil
D. palms covered with lotion
Answer:


In [7]:
def format_prompt(prompt):
    
    return tokenizer.apply_chat_template([
        {"role": "user",
         "content": prompt}], 
        tokenize=False, add_generation_prompt=True
    )
    
print(format_prompt(build_prompt(full_set.iloc[0])))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Select only one option from A, B, C, or D to answer the following multiple-choice question. Do not output anything else.
Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
A. dry palms
B. wet palms
C. palms covered with oil
D. palms covered with lotion
Answer:<|im_end|>
<|im_start|>assistant



In [8]:
def run_logits(prompt):
    text = format_prompt(prompt)
    inputs = tokenizer([text], return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask).logits
        
    return logits

sample_logits = run_logits(build_prompt(full_set.iloc[0]))

print(f"Shape: {sample_logits.shape}")
print(f"Logits: {sample_logits}")

Shape: torch.Size([1, 94, 151936])
Logits: tensor([[[ 5.1875,  5.0000,  4.5312,  ..., -5.7188, -5.7188, -5.7188],
         [ 7.7812,  4.6250,  2.0625,  ..., -3.0312, -3.0312, -3.0312],
         [ 8.0000, 10.9375, 10.1250,  ..., -3.8438, -3.8438, -3.8438],
         ...,
         [ 3.5781,  4.7188, -2.7031,  ..., -0.3262, -0.3262, -0.3242],
         [ 9.3750,  3.3438,  3.2031,  ...,  0.3867,  0.3867,  0.3867],
         [ 8.3750, 11.4375,  9.5625,  ..., -1.3828, -1.3828, -1.3828]]],
       device='cuda:0', dtype=torch.bfloat16)


Here 1 means one input passed to model. (batch size)
94 means input token size (94).
151936 means the vocab size of the model.

So this is a 3D tensor, that provides the logits for the next token for each of the 94 input tokens.

For example, logits[0, 0, :] is the score for predicting next token after position 0.


In [9]:
final_token_logits = sample_logits[:, -1, :]
print(f"Final token logits shape: {final_token_logits.shape}")
print(f"Final token logits: {final_token_logits}")

Final token logits shape: torch.Size([1, 151936])
Final token logits: tensor([[ 8.3750, 11.4375,  9.5625,  ..., -1.3828, -1.3828, -1.3828]],
       device='cuda:0', dtype=torch.bfloat16)


In [10]:
import matplotlib.pyplot as plt
import numpy as np

def get_token_and_prob_full_vocab(final_logits, plot=False):
    
    probs = torch.softmax(final_logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k=10, dim=-1)
    
    # lets plot the probs
    
    if plot:
        
        probs_np = probs.detach().cpu().numpy().flatten()
        x = range(len(probs_np))
        
        plt.figure(figsize=(8, 4))
        plt.bar(x, probs_np, alpha=0.6, label='Probability (bar)')
        plt.plot(x, probs_np, color='red', marker='o', linewidth=1.5, label='Distribution (line)')
        
        plt.yscale('log')
        
        plt.xlabel('Class / Token index')
        plt.ylabel('Probability')
        plt.title('Softmax Probability Distribution')
        plt.legend()
        plt.tight_layout()
        plt.show()
    
    results = []
    for prob, token_id in zip(top_probs[0], top_ids[0]):
        token = tokenizer.decode([token_id.item()], skip_special_tokens=True)
        results.append((token, prob.item()))
        
    return results, results[0]


results, top_pair = get_token_and_prob_full_vocab(final_token_logits, plot=False)
print(f"Top Token-Prob Pair: {top_pair}")
print("="*5)
print("Top Ten: ")
for result in results:
    print(result)

Top Token-Prob Pair: ('B', 0.9375)
=====
Top Ten: 
('B', 0.9375)
('b', 0.046630859375)
('The', 0.00262451171875)
('C', 0.0023193359375)
('D', 0.00124359130859375)
('A', 0.00090789794921875)
('Based', 0.00058746337890625)
('dry', 0.00058746337890625)
('It', 0.0004863739013671875)
('Answer', 0.0004558563232421875)


In [11]:
def get_option_logits(final_logits, tokenizer, options=("A", "B", "C", "D")):
    
    option_ids = []
    for opt in options:
        
        ids = tokenizer.encode(opt, add_special_tokens=False)
            
        option_ids.append(ids[0])
        
    return option_ids

ids = get_option_logits(final_token_logits, tokenizer)
print(ids)

[32, 33, 34, 35]


In [12]:
def softmax_over_options_only(final_logits, tokenizer, options=("A", "B", "C", "D")):
    
    option_ids = get_option_logits(final_logits, tokenizer, options)
    restricted_logits = final_logits[:, option_ids]
    
    probs = torch.softmax(restricted_logits, dim=-1)
    top_prob, top_idx = torch.max(probs, dim=-1)
    
    best = (options[top_idx[0].item()], top_prob[0].item())
    return best

print(softmax_over_options_only(final_token_logits, tokenizer))

('B', 0.99609375)


In [13]:
def softmax_then_options(final_logits, tokenizer, options=("A", "B", "C", "D")):
    
    option_ids = get_option_logits(final_logits, tokenizer, options)
    probs = torch.softmax(final_logits, dim=-1)
    option_probs = probs[:, option_ids]
    
    top_prob, top_idx = torch.max(option_probs, dim=-1)
    
    best = (options[top_idx[0].item()], top_prob[0].item())
    return best

print(softmax_then_options(final_token_logits, tokenizer))

('B', 0.9375)


In [ ]:
from tqdm import tqdm

def arc_inference(n_rows=None):
    
    set = None
    
    if n_rows:
        set = full_set.head(n_rows)
    else:
        set = full_set
        
    vocab_options_correct = 0
    vocab_options_prob = 0
    
    softmax_options_correct = 0
    softmax_options_prob = 0
    
    softmax_and_options_correct = 0
    softmax_and_options_prob = 0
    
    for _, row in tqdm(set.iterrows(), total=len(set)):
        
        prompt = build_prompt(row)
        prompt = format_prompt(prompt)
        logits = run_logits(prompt)
        final_token_logits = logits[:, -1, :]
    
        _, full_vocab_result = get_token_and_prob_full_vocab(final_token_logits, plot=False)
        options_only = softmax_over_options_only(final_token_logits, tokenizer)
        hybrid = softmax_then_options(final_token_logits, tokenizer)
        
        answer_option = row["answer_key"]
        
        if full_vocab_result[0] == answer_option:
            vocab_options_correct += 1
            vocab_options_prob += full_vocab_result[1]
            
        if options_only[0] == answer_option:
            softmax_options_correct += 1
            softmax_options_prob += options_only[1]
            
        if hybrid[0] == answer_option:
            softmax_and_options_correct += 1
            softmax_and_options_prob += hybrid[1]
            
    print(f"Accuracy of full vocab: {vocab_options_correct / len(set):.3f}")
    print(f"Mean correct probability: {vocab_options_prob / vocab_options_correct:.3f}")
            
    print(f"Accuracy of only options: {softmax_options_correct / len(set):.3f}")
    print(f"Mean correct probability: {softmax_options_prob / softmax_options_correct:.3f}")
            
    print(f"Accuracy of hybrid: {softmax_and_options_correct / len(set):.3f}")
    print(f"Mean correct probability: {softmax_and_options_prob / softmax_and_options_correct:.3f}")
            

arc_inference(n_rows=None)

100%|██████████| 2582/2582 [02:36<00:00, 16.50it/s]

Accuracy of full vocab: 0.674
Mean correct probability: 0.835
Accuracy of only options: 0.677
Mean correct probability: 0.904
Accuracy of full vocab: 0.677
Mean correct probability: 0.833
